# 06 — Layer A FAISS 全体再ビルド

**目的**: macOS CPU 環境では e5-large の推論メモリが不足するため、  
Colab の GPU (T4/A100) を使って Layer A FAISS インデックスを完全再ビルドする。

## 実行前チェックリスト
- [ ] ランタイムタイプが **GPU (T4 以上)** になっていることを確認
  - メニュー → ランタイム → ランタイムのタイプを変更 → T4 GPU
- [ ] Google Drive を接続する場合はセル 2 を実行する
- [ ] ファイルを手動アップロードする場合は `UPLOAD_MODE = True` を設定する

## ビルド内容
| 項目 | 値 |
|------|----|
| モデル | `intfloat/multilingual-e5-large` |
| ベクトル次元 | 1024 |
| 対象レコード | 2,231 件（既存 2,040 + regulation 191） |
| インデックス型 | `IndexFlatIP`（内積・正規化済み → コサイン類似度） |
| 出力ファイル | `layer_a.index` + `layer_a_meta.json` |

## ソース内訳
| source_type | 件数 | 内容 |
|-------------|------|------|
| law | 944 | 貨物等省令 Article/Item チャンク |
| parameter | 406 | 技術パラメータ（数値閾値） |
| tsutatsu | 53 | 輸出管理令 通達 |
| eccn | 637 | ECCN Part774 エントリ |
| regulation | **191** | **control_nodes.json 規制ノード（今回追加）** |

---
## ステップ 0: GPU 確認

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ GPU: {result.stdout.strip()}')
else:
    print('❌ GPU not found! ランタイムを GPU に変更してください。')
    raise RuntimeError('GPU required')

import torch
print(f'   PyTorch CUDA: {torch.cuda.is_available()}')
print(f'   CUDA device: {torch.cuda.get_device_name(0)}')

---
## ステップ 1: 依存関係インストール

In [ ]:
%%capture install_log
!pip install -q faiss-gpu sentence-transformers==3.3.1 numpy
print('✅ インストール完了')

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

print(f'faiss version: {faiss.__version__}')
print(f'faiss GPU support: {faiss.get_num_gpus() > 0}')

---
## ステップ 2: データファイルの準備

**方法 A: Google Drive からマウント**（リポジトリが Drive に同期されている場合）  
**方法 B: 手動アップロード**（下記のセルで `UPLOAD_MODE = True` に変更）

In [ ]:
# ──────────────────────────────────────────────────
# 設定: 環境に合わせて変更してください
# ──────────────────────────────────────────────────

# True = ファイルを手動アップロード / False = Google Drive から読み込み
UPLOAD_MODE = False

# Google Drive のリポジトリルートパス (UPLOAD_MODE=False の場合)
DRIVE_REPO_ROOT = '/content/drive/MyDrive/AI_TradeManagement'

# GitHub から clone する場合はこちらを使用
GITHUB_CLONE = False
GITHUB_REPO  = 'https://github.com/tsp0918/AI_TradeManagement.git'
GITHUB_BRANCH = 'branch_neurosymbolic'

# 出力先（生成したファイルを Drive に保存する場合）
SAVE_TO_DRIVE = False
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/AI_TradeManagement/data/staging'

# ──────────────────────────────────────────────────
MODEL_NAME = 'intfloat/multilingual-e5-large'
BATCH_SIZE  = 128   # T4 16GB: 128 が推奨, A100: 256 まで可
DIM         = 1024

WORK_DIR = Path('/content/work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

print('設定完了')

In [ ]:
if not UPLOAD_MODE and not GITHUB_CLONE:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path(DRIVE_REPO_ROOT)
    print(f'✅ Drive マウント完了: {REPO_ROOT}')
elif GITHUB_CLONE:
    import subprocess
    # GitHub Personal Access Token が必要な場合は userinfo に含める
    print('GitHub から clone します...')
    result = subprocess.run(
        ['git', 'clone', '--branch', GITHUB_BRANCH, '--depth', '1', GITHUB_REPO, '/content/repo'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('Clone エラー:', result.stderr)
        raise RuntimeError('git clone failed')
    REPO_ROOT = Path('/content/repo')
    print(f'✅ Clone 完了: {REPO_ROOT}')
else:
    print('⚠️  UPLOAD_MODE=True のため Drive マウントをスキップ')
    print('   次のセルで layer_a_meta.json をアップロードしてください')
    REPO_ROOT = None

In [ ]:
if UPLOAD_MODE:
    from google.colab import files
    print('layer_a_meta.json をアップロードしてください')
    uploaded = files.upload()
    META_PATH = WORK_DIR / 'layer_a_meta.json'
    for fname, content in uploaded.items():
        META_PATH.write_bytes(content)
    print(f'✅ アップロード完了: {META_PATH}')
else:
    META_PATH = REPO_ROOT / 'data' / 'staging' / 'layer_a_meta.json'
    assert META_PATH.exists(), f'ファイルが見つかりません: {META_PATH}'
    print(f'✅ Meta ファイル: {META_PATH}')

# メタデータ読み込み
with META_PATH.open() as f:
    meta = json.load(f)

records = meta['records']
print(f'\n総レコード数: {len(records)}')

from collections import Counter
ct = Counter(r.get('source_type', '?') for r in records)
print('source_type 別件数:')
for k, v in sorted(ct.items()):
    pending_mark = ' ← 今回追加' if k == 'regulation' else ''
    print(f'  {k:15s}: {v:5d}{pending_mark}')

pending = [r for r in records if r.get('_pending_embed')]
print(f'\n_pending_embed フラグ: {len(pending)} 件')

---
## ステップ 3: エンコード戦略の選択

| 戦略 | 内容 | 時間（T4推定） |
|------|------|---------------|
| **A: フル再ビルド** | 全 2,231 件を e5-large で再エンコード | ~4〜6 分 |
| **B: 差分追加** | 既存 index に 191 件のみ追加 | ~0.5 分 |  

**推奨: 戦略 A** — 全件を同じモデル・バージョンで揃えることで一貫性を保つ。  
既存の `layer_a.index` が手元にない場合は戦略 A のみ選択可能。

In [ ]:
# ──────────────────────────────────────────────────
# 戦略選択
# ──────────────────────────────────────────────────

STRATEGY = 'A'   # 'A' = フル再ビルド / 'B' = 差分追加

print(f'戦略: {STRATEGY} — ', end='')
if STRATEGY == 'A':
    print('フル再ビルド（全 2,231 件）')
    RECORDS_TO_EMBED = records
elif STRATEGY == 'B':
    print('差分追加（regulation 191 件のみ）')
    RECORDS_TO_EMBED = [r for r in records if r.get('_pending_embed')]
    print(f'   → エンコード対象: {len(RECORDS_TO_EMBED)} 件')
    # 既存 index が必要
    if not UPLOAD_MODE:
        EXISTING_INDEX_PATH = REPO_ROOT / 'data' / 'staging' / 'layer_a.index'
    else:
        EXISTING_INDEX_PATH = WORK_DIR / 'layer_a.index'
        print('⚠️  layer_a.index もアップロードが必要です（次のセルで実行）')
else:
    raise ValueError(f'STRATEGY は A または B にしてください')

In [ ]:
# 戦略 B かつ UPLOAD_MODE の場合: layer_a.index をアップロード
if STRATEGY == 'B' and UPLOAD_MODE:
    from google.colab import files
    print('layer_a.index をアップロードしてください')
    uploaded = files.upload()
    for fname, content in uploaded.items():
        (WORK_DIR / 'layer_a.index').write_bytes(content)
    print('✅ layer_a.index アップロード完了')
else:
    print('（このセルはスキップ）')

---
## ステップ 4: モデルのロードとエンコード

In [ ]:
print(f'モデルをロード中: {MODEL_NAME}')
t0 = time.time()
model = SentenceTransformer(MODEL_NAME, device='cuda')
print(f'✅ ロード完了 ({time.time()-t0:.1f}s)')
print(f'   埋め込み次元: {model.get_sentence_embedding_dimension()}')

# GPU メモリ確認
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv,noheader

In [ ]:
# embed_text フィールドを抽出
embed_texts = [r.get('embed_text', '') for r in RECORDS_TO_EMBED]

# 空テキストのチェック
empty_count = sum(1 for t in embed_texts if not t.strip())
if empty_count:
    print(f'⚠️  embed_text が空のレコード: {empty_count} 件（スキップされます）')

print(f'エンコード開始: {len(embed_texts)} 件 / batch_size={BATCH_SIZE}')
t0 = time.time()

embeddings = model.encode(
    embed_texts,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,   # IndexFlatIP でコサイン類似度として機能
    show_progress_bar=True,
    convert_to_numpy=True,
)

elapsed = time.time() - t0
vectors = np.asarray(embeddings, dtype='float32')
print(f'\n✅ エンコード完了: {elapsed:.1f}s')
print(f'   ベクトル形状: {vectors.shape}')
print(f'   ノルム確認（先頭3件）: {np.linalg.norm(vectors[:3], axis=1)}')

---
## ステップ 5: FAISS インデックス構築

In [ ]:
if STRATEGY == 'A':
    # フル再ビルド: 新しいインデックスを作成
    print('フル再ビルド: 新規 IndexFlatIP を作成...')
    
    # GPU インデックスを使って高速ビルド
    res = faiss.StandardGpuResources()
    gpu_index = faiss.GpuIndexFlatIP(res, DIM)
    gpu_index.add(vectors)
    
    # CPU インデックスに転送（保存・ロードは CPU のみ対応）
    cpu_index = faiss.index_gpu_to_cpu(gpu_index)
    print(f'✅ インデックス構築完了: ntotal={cpu_index.ntotal}')

elif STRATEGY == 'B':
    # 差分追加: 既存インデックスをロードして追加
    print('差分追加: 既存インデックスをロード...')
    existing_index_path = str(EXISTING_INDEX_PATH)
    cpu_index = faiss.read_index(existing_index_path)
    print(f'   既存: ntotal={cpu_index.ntotal}')
    
    cpu_index.add(vectors)
    print(f'✅ 追加完了: ntotal={cpu_index.ntotal} (+{len(RECORDS_TO_EMBED)})')

In [ ]:
# ── 動作確認: サンプルクエリで検索 ────────────────────────────────────────

def search_index(query: str, top_k: int = 5):
    q = model.encode([f'query: {query}'], normalize_embeddings=True)
    q_np = np.asarray(q, dtype='float32')
    D, I = cpu_index.search(q_np, top_k)
    print(f'\nQuery: "{query}"')
    for rank, (score, idx) in enumerate(zip(D[0], I[0])):
        if idx < 0: continue
        rec = RECORDS_TO_EMBED[idx] if STRATEGY == 'A' else records[idx]
        title = rec.get('title') or rec.get('label') or rec.get('category') or ''
        src   = rec.get('source_type', '')
        iname = rec.get('item_no', '') or rec.get('node_id', '')
        print(f'  [{rank+1}] score={score:.4f} | {src} | {iname} | {title[:50]}')

# テストクエリ
search_index('工作機械の位置決め精度の規制要件')
search_index('nuclear reactor uranium enrichment')
search_index('半導体製造装置 ECCN 規制')
search_index('ミサイルロケット推進装置')

---
## ステップ 6: メタデータ更新と保存

In [ ]:
from collections import Counter

# メタデータを更新
final_records = RECORDS_TO_EMBED  # フル再ビルドの場合は全レコード
if STRATEGY == 'B':
    final_records = records  # 差分の場合は全レコード（既存 + 新規）

# faiss_id を再割り当て + _pending_embed フラグを削除
for i, rec in enumerate(final_records):
    rec['faiss_id'] = i
    rec.pop('_pending_embed', None)  # フラグを削除（埋め込み済み）

src_breakdown = dict(Counter(r.get('source_type', '?') for r in final_records))

new_meta = {
    'total': cpu_index.ntotal,
    'dim': DIM,
    'model': MODEL_NAME,
    'built_at': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'source_breakdown': src_breakdown,
    'rebuild_notes': (
        'Full rebuild via Colab GPU (2026-03-21). '
        'Added 191 regulation nodes from control_nodes.json. '
        f'Strategy: {STRATEGY}. Model: {MODEL_NAME}.'
    ),
    'records': final_records,
}

print('更新後のメタデータ:')
print(f'  total:  {new_meta["total"]}')
print(f'  dim:    {new_meta["dim"]}')
print(f'  source_breakdown: {src_breakdown}')
print(f'  records: {len(final_records)}')

In [ ]:
# 作業ディレクトリに保存
OUT_INDEX = WORK_DIR / 'layer_a.index'
OUT_META  = WORK_DIR / 'layer_a_meta.json'

faiss.write_index(cpu_index, str(OUT_INDEX))
with OUT_META.open('w', encoding='utf-8') as f:
    json.dump(new_meta, f, ensure_ascii=False)

print(f'✅ 保存完了')
print(f'   layer_a.index : {OUT_INDEX.stat().st_size / 1024 / 1024:.1f} MB')
print(f'   layer_a_meta.json: {OUT_META.stat().st_size / 1024:.0f} KB')

In [ ]:
# Google Drive にも保存する場合
if SAVE_TO_DRIVE:
    import shutil
    drive_out = Path(DRIVE_OUTPUT_DIR)
    drive_out.mkdir(parents=True, exist_ok=True)
    shutil.copy(OUT_INDEX, drive_out / 'layer_a.index')
    shutil.copy(OUT_META,  drive_out / 'layer_a_meta.json')
    print(f'✅ Drive にコピー完了: {drive_out}')
else:
    print('（Drive 保存スキップ: SAVE_TO_DRIVE=False）')

---
## ステップ 7: ファイルのダウンロード

生成した `layer_a.index` と `layer_a_meta.json` をローカルにダウンロードし、  
`data/staging/` に配置します。

In [ ]:
from google.colab import files

print('layer_a.index をダウンロードします...')
files.download(str(OUT_INDEX))

print('layer_a_meta.json をダウンロードします...')
files.download(str(OUT_META))

print('\n✅ ダウンロード完了')
print('\n次のステップ:')
print('1. ダウンロードしたファイルを data/staging/ に配置')
print('2. platform-core を再起動（FAISSは起動時にロード）')
print('3. GET /api/faiss/search/layer-a?q=テスト で動作確認')

---
## ステップ 8: 最終確認（ローカル配置後）

ダウンロードしたファイルを `data/staging/` に置いた後、  
以下のコマンドでローカル確認できます:

```python
import faiss, json
idx = faiss.read_index('data/staging/layer_a.index')
meta = json.load(open('data/staging/layer_a_meta.json'))
print('ntotal:', idx.ntotal)  # → 2231
print('records:', len(meta['records']))  # → 2231
print('breakdown:', meta['source_breakdown'])
# → {'fefta_law': 944, 'fefta_parameter': 406, 'fefta_tsutatsu': 53, 'eccn': 637, 'regulation': 191}
```

### DAP RAG 動作確認

platform-core 起動後:
```bash
curl -s 'http://localhost:8000/api/faiss/search/layer-a?q=工作機械規制&top_k=3' | python3 -m json.tool
```

---
## 付録: トラブルシューティング

### OOM (Out of Memory) が発生する場合
`BATCH_SIZE` を小さくしてください:
```python
BATCH_SIZE = 64  # T4 8GB 環境
BATCH_SIZE = 32  # さらに少ないメモリ環境
```

### `faiss-gpu` のインストールが失敗する場合
```bash
!pip install faiss-cpu  # CPU 版で代替
```
その後 `GpuIndexFlatIP` → `IndexFlatIP` に変更してください。

### sentence-transformers のバージョンエラー
```bash
!pip install sentence-transformers==2.7.0
```

### Drive のパスが見つからない
- Drive マウント後 `/content/drive/MyDrive/` 以下のパスを確認
- `!ls /content/drive/MyDrive/` で確認できます